In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_de = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/de_2007_2015_bn.csv")

In [3]:
df_de = df_de.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_de

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,PMSI,JI,PR,DWCP,DBPG,EUPC
0,1.46,6.55,6.71,2.26,236.0,-0.13,26.46,20.787343,17.555365,3.027782,...,72.864286,0.11,15719.0,-28436.0,50.0,0.952,90,82.709673,0.0,6.196333e+06
1,0.81,6.54,6.67,1.78,236.0,-0.19,26.06,20.898953,17.951098,1.102460,...,72.864286,0.00,5641.0,2834.0,50.0,0.952,90,82.821877,0.0,1.001268e+07
2,1.63,6.49,6.61,1.68,235.0,-0.25,23.87,18.036010,19.597935,-5.305528,...,67.800000,0.02,3068.0,-56474.0,50.0,0.952,90,82.758552,0.0,3.799936e+07
3,2.48,6.42,6.61,1.65,235.0,-0.15,26.06,19.555668,19.176892,4.306441,...,68.800000,0.04,7319.0,30446.0,50.0,0.952,90,82.888440,0.0,3.479371e+07
4,2.55,6.15,6.45,1.64,230.0,-1.85,26.31,21.292775,18.693432,5.699283,...,70.300000,-0.34,22139.0,-29687.0,50.0,0.952,90,82.852483,0.0,3.985701e+07
5,1.82,6.09,6.35,1.56,231.0,0.19,26.58,19.504103,18.897792,0.285850,...,70.700000,0.24,14921.0,13203.0,50.0,0.952,90,82.818229,0.0,4.092093e+07
6,1.76,6.01,6.08,1.45,231.0,0.27,25.91,19.810761,19.247998,0.112644,...,71.500000,0.42,8054.0,16374.0,50.0,0.952,90,82.870361,0.0,5.269857e+07
7,0.44,5.88,5.94,1.43,232.0,0.42,26.01,20.158215,19.210129,1.744464,...,72.200000,0.27,13598.0,-40820.0,50.0,0.952,90,82.919183,0.0,3.929514e+07
8,1.82,5.72,5.95,1.39,234.0,0.87,25.92,19.461799,19.312236,0.776508,...,72.700000,0.24,26869.0,4675.0,50.0,0.952,90,82.829850,0.0,2.798182e+07


In [5]:
discretised_de = pd.DataFrame(index=df_de.index)

for col in df_de.columns:
    no_unique = df_de[col].nunique()
    
    if no_unique <= 1:
        discretised_de[col] = 0
    else:
        try:
            discretised_de[col] = pd.qcut(
                df_de[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_de[col] = df_de[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_de)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    0    2    2     2     2    1     2     2     0      2  ...    2    1   
1    0    2    2     2     2    0     1     2     0      1  ...    2    0   
2    1    2    1     2     1    0     0     0     2      0  ...    0    0   
3    2    1    1     1     1    1     1     1     1      2  ...    0    1   
4    2    1    1     1     0    0     2     2     0      2  ...    0    0   
5    1    1    1     1     0    1     2     0     1      0  ...    1    1   
6    1    0    0     0     0    2     0     1     2      0  ...    1    2   
7    0    0    0     0     1    2     1     1     1      1  ...    1    2   
8    1    0    0     0     1    2     0     0     2      1  ...    2    1   

   EPFRG  GGAG  PMSI  JI  PR  DWCP  DBPG  EUPC  
0      2     1     0   0   0     0     0     0  
1      0     1     0   0   0     1     0     0  
2      0     0     0   0   0     0     0     1  
3      0     2 

In [6]:
sm = from_pandas(discretised_de)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_de.html")

Graphs/01_fully_connected_de.html


In [8]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_de.html")

Graphs/01_thresholded_de.html


In [9]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_de_1.html")

In [10]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)

     From     To    Weight
6     QOA   GGAG  5.482937
9    EIAP   GGFC  2.377894
23   EBSG  GDPCG  2.334403
3     QOA   VABI  2.239349
7     QOA   DWCP  1.946871
38    EMP   GGFC  1.552538
8    EIAP   GCFP  1.455224
27   GDPC    FDI  1.444933
1     FDI  GDPCG  1.414518
0     FDI    QOR  1.222382
29   GDPC   VABI  1.220303
21   EBSG   VABI  1.187675
18   GGFC   GGAG  1.179111
33   LBGD   GCFP  1.160727
36   LBGD   DWCP  1.128558
19  GDPCG   GCFP  1.097556
37    EMP    FDI  1.058044
42    LFG   GGFC  1.035589
14   PDPS   DWCP  1.017250
40    FEM   GGFC  1.016469
30    EIS   GDPC  1.001479
44    LFG    EIS  0.999994
2     QOR    EII  0.989422
41    FEM    EMP  0.985860
35   LBGD    LTU  0.960247
43    LFG   EBSG  0.916121
25   EBSG  EPFRG  0.902816
20    EII   EIAP  0.869114
24   EBSG    FEM  0.820469
16   VABI  GDPCG -0.819994
26   EBSG   DWCP -0.824448
12   PDPS   VABI -0.894217
28   GDPC    QOR -1.011353
17   GGFC  GDPCG -1.169420
39    EMP    LTU -1.252944
32   LBGD   VABI -1.301259
5